In [ ]:
# Colab setup
import sys
if "google.colab" in sys.modules:
    %pip install -q pyedautils ephem statsmodels plotly kaleido ipywidgets

import plotly.io as pio
pio.renderers.default = "notebook_connected"

# Mollier h-x Diagram

```{admonition} Goal
:class: tip
Display measured indoor conditions on a Mollier h-x (psychrometric) diagram — the relationship between temperature, humidity and enthalpy of moist air, with a comfort zone, dashed tolerance bands and frequency contour lines.
```

```{admonition} Data Basis
:class: note
We use the bundled dataset `room_climate_sample.csv` shipped with **pyedautils** —
anonymised hourly temperature and humidity from a real sensor in a cool,
humidity-controlled room. It is loaded directly from the installed package
(`importlib.resources`), so no download is needed.
```

## Data Preparation

In [ ]:
import pandas as pd
from importlib import resources

data_path = resources.files("pyedautils") / "data" / "room_climate_sample.csv"
df = pd.read_csv(data_path)   # columns: timestamp, humidity, temperature
df.head()

## Visualization

Pass the DataFrame to `plot_mollier_hx`. Measured points are coloured by season;
the default **comfort zone** (T 20–26 °C, φ 30–65 %) is drawn as the green polygon.
All charts are interactive — hover a point for details.

Every diagram below is drawn for the building's **site altitude** (here 450 m a.s.l.):
the matching air pressure comes from `pressure_from_altitude(...)`, so the psychrometric
relations are correct for the thinner air. Altitude and pressure are written in the
top-right corner.

In [ ]:
from pyedautils.plots import plot_mollier_hx
from pyedautils._mollier import pressure_from_altitude
from IPython.display import HTML, display

# Site altitude — the air pressure follows from it, so the psychrometric relations
# match the thinner air. Defined once and reused by every chart below.
altitude = 450  # m a.s.l.
pressure = pressure_from_altitude(altitude)

html = plot_mollier_hx(df, altitude=altitude, pressure=pressure, height=700)
display(HTML(html))

## Comfort zone & tolerance bands

Two **dashed tolerance bands** sit on top of the green comfort zone via
`comfort_zone_orange` and `comfort_zone_red`. Each takes the same absolute-bounds
dict as `comfort_zone` — `temperature` (°C), `rel_humidity` (0–1) and `abs_humidity`
(kg/kg, e.g. `(0, 0.012)` for a 12 g/kg ceiling) — plus a `label` and `color`. The
`abs_humidity` cap closes every zone on the right, so air above the ceiling is flagged
even when its relative humidity still looks fine.

```{admonition} Tolerance bands
:class: note
The bands below grade **± 1 K / 5 % / 12 g/kg** (orange) and **± 2.5 K / 10 % / 13 g/kg**
(red) around the comfort zone, which itself caps absolute humidity at 11.5 g/kg. All
three caps are shown in the legend.
```

In [ ]:
html = plot_mollier_hx(
    df,
    altitude=altitude, pressure=pressure,
    comfort_zone={"temperature": (20, 26), "rel_humidity": (0.30, 0.65),
                  "abs_humidity": (0, 0.0115)},
    comfort_label="Comfort zone (≤ 11.5 g/kg)",
    comfort_zone_orange={"temperature": (19, 27), "rel_humidity": (0.25, 0.70),
                         "abs_humidity": (0, 0.012),
                         "label": "± 1 K / 5 % / 12 g/kg"},
    comfort_zone_red={"temperature": (17.5, 28.5), "rel_humidity": (0.20, 0.75),
                      "abs_humidity": (0, 0.013),
                      "label": "± 2.5 K / 10 % / 13 g/kg"},
    height=700,
)
display(HTML(html))

## Frequency contour lines

With enough data, `show_frequency=True` overlays cumulative-frequency contour lines:
each line labelled *N* encloses all but *N* units per year, so the outer lines mark
the rare extremes and the inner lines the typical core. Use `frequency_unit="days"`
to count distinct calendar days instead of hours.

Here we switch to a full **outdoor weather reference year for Zürich-Kloten** (bundled
with pyedautils) — the annual spread from cold, dry winters to warm, humid summers is
exactly what the contours are meant to show. The comfort zone is hidden
(`comfort_zone=False`), as outdoor weather is not about indoor comfort.

In [ ]:
df_kloten = pd.read_csv(
    resources.files("pyedautils") / "data" / "kloten_weather_sample.csv")

html = plot_mollier_hx(df_kloten, altitude=altitude, pressure=pressure,
                       comfort_zone=False,
                       show_frequency=True, frequency_unit="hours", height=700)
display(HTML(html))

## Customisation

::::{dropdown} Axis ranges, altitude, curve families & convention
- `domain_x` / `domain_y` zoom into a region — absolute humidity [kg/kg] and the
  y-coordinate (≈ temperature at x = 0).
- `altitude` / `pressure` set the site altitude (m a.s.l., shown top-right); pass
  `pressure_from_altitude(...)` so the psychrometric relations match the thinner air.
- `show_temperature`, `show_density`, `show_rel_humidity`, `show_enthalpy`,
  `show_abs_humidity` toggle each iso-line family individually.
- `convention` switches the coordinate normalisation between `'classical'`
  (default, Recknagel) and `'glueck'`.
- `height` sets the diagram height in pixels.

```python
from pyedautils._mollier import pressure_from_altitude

html = plot_mollier_hx(
    df,
    altitude=1500,                            # site altitude, shown top-right
    pressure=pressure_from_altitude(1500),    # matching air pressure
    domain_x=(0.002, 0.014),   # abs. humidity 2–14 g/kg
    domain_y=(10, 35),         # y-range ≈ 10–35 °C
    show_density=False,
    convention="classical",
    height=700,
)
```
::::